In [4]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

class BrainGCN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2): 
        super(BrainGCN, self).__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        edge_weight = edge_attr.squeeze() if edge_attr is not None else None
        
        x = self.conv1(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

class BrainGIN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2):
        super(BrainGIN, self).__init__()
        nn1 = Sequential(Linear(in_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(nn1)
        
        nn2 = Sequential(Linear(hidden_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv2 = GINConv(nn2)
        
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

def load_v8_data(data_dir):
    print(f" [{data_dir}] 경로에서 V8 전체 데이터 로드 중...")
    data_list = []
    
    for file_name in os.listdir(data_dir):
        if file_name.startswith('sub_'): 
            try:
                
                data = torch.load(os.path.join(data_dir, file_name), map_location='cpu', weights_only=False)
                data_list.append(data)
            except Exception as e:
                print(f" {file_name} 로드 실패: {e}")
                continue
                
    print(f"✅ 총 {len(data_list)}명 전체 데이터 로드 완료!\n")
    return data_list

def evaluate_model(data_list, model_type="GCN", epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    labels = [int(data.y.item()) for data in data_list]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    acc_scores, f1_scores = [], []
    print(f" [{model_type}] 모델 5-Fold 교차 검증 시작 ")

    for fold, (train_idx, test_idx) in enumerate(skf.split(data_list, labels)):
        train_data = [data_list[i] for i in train_idx]
        test_data = [data_list[i] for i in test_idx]
        
        train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
        
        input_dim = data_list[0].num_features 
        
        if model_type == "GCN":
            model = BrainGCN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
        elif model_type == "GIN":
            model = BrainGIN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
            
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = torch.nn.CrossEntropyLoss()
        
        # 학습
        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()
                
        # 평가
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                preds.extend(out.argmax(dim=1).cpu().numpy())
                trues.extend(batch.y.cpu().numpy())
                
        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds, average='macro')
        acc_scores.append(acc)
        f1_scores.append(f1)
        print(f"Fold {fold+1} | Accuracy: {acc:.4f} | F1 Score: {f1:.4f}")
        
    print(f" [{model_type}] 최종 평균 -> Accuracy: {np.mean(acc_scores):.4f} | F1: {np.mean(f1_scores):.4f}\n")
    return np.mean(acc_scores), np.mean(f1_scores)

if __name__ == "__main__":
    V8_DATA_DIR = "data_AAL_Adolescent_Top20"

    all_data = load_v8_data(V8_DATA_DIR)

    gcn_acc, gcn_f1 = evaluate_model(all_data, model_type="GCN", epochs=50)
    
    gin_acc, gin_f1 = evaluate_model(all_data, model_type="GIN", epochs=50)
    
    print("====================================================")
    print(f"- GCN | Accuracy: {gcn_acc:.4f} | F1: {gcn_f1:.4f}")
    print(f"- GIN | Accuracy: {gin_acc:.4f} | F1: {gin_f1:.4f}")
    print("====================================================")

 [data_AAL_Adolescent_Top20] 경로에서 V8 전체 데이터 로드 중...
✅ 총 827명 전체 데이터 로드 완료!

 [GCN] 모델 5-Fold 교차 검증 시작 
Fold 1 | Accuracy: 0.5422 | F1 Score: 0.3516
Fold 2 | Accuracy: 0.5422 | F1 Score: 0.3516
Fold 3 | Accuracy: 0.5394 | F1 Score: 0.3504
Fold 4 | Accuracy: 0.5455 | F1 Score: 0.3529
Fold 5 | Accuracy: 0.5455 | F1 Score: 0.3529
 [GCN] 최종 평균 -> Accuracy: 0.5429 | F1: 0.3519

 [GIN] 모델 5-Fold 교차 검증 시작 
Fold 1 | Accuracy: 0.5904 | F1 Score: 0.5882
Fold 2 | Accuracy: 0.5843 | F1 Score: 0.5672
Fold 3 | Accuracy: 0.5273 | F1 Score: 0.4993
Fold 4 | Accuracy: 0.4970 | F1 Score: 0.4718
Fold 5 | Accuracy: 0.5394 | F1 Score: 0.4273
 [GIN] 최종 평균 -> Accuracy: 0.5477 | F1: 0.5108

- GCN | Accuracy: 0.5429 | F1: 0.3519
- GIN | Accuracy: 0.5477 | F1: 0.5108


In [5]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

class BrainGCN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2): 
        super(BrainGCN, self).__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        edge_weight = edge_attr.squeeze() if edge_attr is not None else None
        
        x = self.conv1(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

class BrainGIN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2):
        super(BrainGIN, self).__init__()
        nn1 = Sequential(Linear(in_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(nn1)
        
        nn2 = Sequential(Linear(hidden_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv2 = GINConv(nn2)
        
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

def load_v8_data(data_dir):
    print(f" [{data_dir}] 경로에서 V8 전체 데이터 로드 중...")
    data_list = []
    
    for file_name in os.listdir(data_dir):
        if file_name.startswith('sub_'): 
            try:
                
                data = torch.load(os.path.join(data_dir, file_name), map_location='cpu', weights_only=False)
                data_list.append(data)
            except Exception as e:
                print(f" {file_name} 로드 실패: {e}")
                continue
                
    print(f"✅ 총 {len(data_list)}명 전체 데이터 로드 완료!\n")
    return data_list

def evaluate_model(data_list, model_type="GCN", epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    labels = [int(data.y.item()) for data in data_list]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    acc_scores, f1_scores = [], []
    print(f" [{model_type}] 모델 5-Fold 교차 검증 시작 ")

    for fold, (train_idx, test_idx) in enumerate(skf.split(data_list, labels)):
        train_data = [data_list[i] for i in train_idx]
        test_data = [data_list[i] for i in test_idx]
        
        train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
        
        input_dim = data_list[0].num_features 
        
        if model_type == "GCN":
            model = BrainGCN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
        elif model_type == "GIN":
            model = BrainGIN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
            
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = torch.nn.CrossEntropyLoss()
        
        # 학습
        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()
                
        # 평가
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                preds.extend(out.argmax(dim=1).cpu().numpy())
                trues.extend(batch.y.cpu().numpy())
                
        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds, average='macro')
        acc_scores.append(acc)
        f1_scores.append(f1)
        print(f"Fold {fold+1} | Accuracy: {acc:.4f} | F1 Score: {f1:.4f}")
        
    print(f" [{model_type}] 최종 평균 -> Accuracy: {np.mean(acc_scores):.4f} | F1: {np.mean(f1_scores):.4f}\n")
    return np.mean(acc_scores), np.mean(f1_scores)

if __name__ == "__main__":
    V8_DATA_DIR = "data_AAL_Adult_Top20"

    all_data = load_v8_data(V8_DATA_DIR)

    gcn_acc, gcn_f1 = evaluate_model(all_data, model_type="GCN", epochs=50)
    
    gin_acc, gin_f1 = evaluate_model(all_data, model_type="GIN", epochs=50)
    
    print("====================================================")
    print(f"- GCN | Accuracy: {gcn_acc:.4f} | F1: {gcn_f1:.4f}")
    print(f"- GIN | Accuracy: {gin_acc:.4f} | F1: {gin_f1:.4f}")
    print("====================================================")

 [data_AAL_Adult_Top20] 경로에서 V8 전체 데이터 로드 중...
✅ 총 258명 전체 데이터 로드 완료!

 [GCN] 모델 5-Fold 교차 검증 시작 
Fold 1 | Accuracy: 0.5385 | F1 Score: 0.3500
Fold 2 | Accuracy: 0.5385 | F1 Score: 0.3500
Fold 3 | Accuracy: 0.5577 | F1 Score: 0.3580
Fold 4 | Accuracy: 0.5490 | F1 Score: 0.3544
Fold 5 | Accuracy: 0.5490 | F1 Score: 0.3544
 [GCN] 최종 평균 -> Accuracy: 0.5465 | F1: 0.3534

 [GIN] 모델 5-Fold 교차 검증 시작 
Fold 1 | Accuracy: 0.4038 | F1 Score: 0.3759
Fold 2 | Accuracy: 0.4038 | F1 Score: 0.4036
Fold 3 | Accuracy: 0.5577 | F1 Score: 0.5495
Fold 4 | Accuracy: 0.6078 | F1 Score: 0.6077
Fold 5 | Accuracy: 0.5686 | F1 Score: 0.4806
 [GIN] 최종 평균 -> Accuracy: 0.5084 | F1: 0.4835

- GCN | Accuracy: 0.5465 | F1: 0.3534
- GIN | Accuracy: 0.5084 | F1: 0.4835


In [6]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

class BrainGCN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2): 
        super(BrainGCN, self).__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        edge_weight = edge_attr.squeeze() if edge_attr is not None else None
        
        x = self.conv1(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

class BrainGIN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2):
        super(BrainGIN, self).__init__()
        nn1 = Sequential(Linear(in_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(nn1)
        
        nn2 = Sequential(Linear(hidden_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv2 = GINConv(nn2)
        
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

def load_v8_data(data_dir):
    print(f" [{data_dir}] 경로에서 V8 전체 데이터 로드 중...")
    data_list = []
    
    for file_name in os.listdir(data_dir):
        if file_name.startswith('sub_'): 
            try:
                
                data = torch.load(os.path.join(data_dir, file_name), map_location='cpu', weights_only=False)
                data_list.append(data)
            except Exception as e:
                print(f" {file_name} 로드 실패: {e}")
                continue
                
    print(f"✅ 총 {len(data_list)}명 전체 데이터 로드 완료!\n")
    return data_list

def evaluate_model(data_list, model_type="GCN", epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    labels = [int(data.y.item()) for data in data_list]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    acc_scores, f1_scores = [], []
    print(f" [{model_type}] 모델 5-Fold 교차 검증 시작 ")

    for fold, (train_idx, test_idx) in enumerate(skf.split(data_list, labels)):
        train_data = [data_list[i] for i in train_idx]
        test_data = [data_list[i] for i in test_idx]
        
        train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
        
        input_dim = data_list[0].num_features 
        
        if model_type == "GCN":
            model = BrainGCN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
        elif model_type == "GIN":
            model = BrainGIN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
            
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = torch.nn.CrossEntropyLoss()
        
        # 학습
        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()
                
        # 평가
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                preds.extend(out.argmax(dim=1).cpu().numpy())
                trues.extend(batch.y.cpu().numpy())
                
        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds, average='macro')
        acc_scores.append(acc)
        f1_scores.append(f1)
        print(f"Fold {fold+1} | Accuracy: {acc:.4f} | F1 Score: {f1:.4f}")
        
    print(f" [{model_type}] 최종 평균 -> Accuracy: {np.mean(acc_scores):.4f} | F1: {np.mean(f1_scores):.4f}\n")
    return np.mean(acc_scores), np.mean(f1_scores)

if __name__ == "__main__":
    V8_DATA_DIR = "data_Schaefer_Adolescent_Top20"

    all_data = load_v8_data(V8_DATA_DIR)

    gcn_acc, gcn_f1 = evaluate_model(all_data, model_type="GCN", epochs=50)
    
    gin_acc, gin_f1 = evaluate_model(all_data, model_type="GIN", epochs=50)
    
    print("====================================================")
    print(f"- GCN | Accuracy: {gcn_acc:.4f} | F1: {gcn_f1:.4f}")
    print(f"- GIN | Accuracy: {gin_acc:.4f} | F1: {gin_f1:.4f}")
    print("====================================================")

 [data_Schaefer_Adolescent_Top20] 경로에서 V8 전체 데이터 로드 중...
✅ 총 827명 전체 데이터 로드 완료!

 [GCN] 모델 5-Fold 교차 검증 시작 
Fold 1 | Accuracy: 0.5422 | F1 Score: 0.3516
Fold 2 | Accuracy: 0.5422 | F1 Score: 0.3516
Fold 3 | Accuracy: 0.5394 | F1 Score: 0.3504
Fold 4 | Accuracy: 0.5455 | F1 Score: 0.3529
Fold 5 | Accuracy: 0.5455 | F1 Score: 0.3529
 [GCN] 최종 평균 -> Accuracy: 0.5429 | F1: 0.3519

 [GIN] 모델 5-Fold 교차 검증 시작 
Fold 1 | Accuracy: 0.5482 | F1 Score: 0.4664
Fold 2 | Accuracy: 0.5361 | F1 Score: 0.4003
Fold 3 | Accuracy: 0.5515 | F1 Score: 0.3996
Fold 4 | Accuracy: 0.4970 | F1 Score: 0.4773
Fold 5 | Accuracy: 0.5455 | F1 Score: 0.4385
 [GIN] 최종 평균 -> Accuracy: 0.5357 | F1: 0.4364

- GCN | Accuracy: 0.5429 | F1: 0.3519
- GIN | Accuracy: 0.5357 | F1: 0.4364


In [7]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

class BrainGCN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2): 
        super(BrainGCN, self).__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        edge_weight = edge_attr.squeeze() if edge_attr is not None else None
        
        x = self.conv1(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

class BrainGIN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2):
        super(BrainGIN, self).__init__()
        nn1 = Sequential(Linear(in_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(nn1)
        
        nn2 = Sequential(Linear(hidden_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv2 = GINConv(nn2)
        
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

def load_v8_data(data_dir):
    print(f" [{data_dir}] 경로에서 V8 전체 데이터 로드 중...")
    data_list = []
    
    for file_name in os.listdir(data_dir):
        if file_name.startswith('sub_'): 
            try:
                
                data = torch.load(os.path.join(data_dir, file_name), map_location='cpu', weights_only=False)
                data_list.append(data)
            except Exception as e:
                print(f" {file_name} 로드 실패: {e}")
                continue
                
    print(f"✅ 총 {len(data_list)}명 전체 데이터 로드 완료!\n")
    return data_list

def evaluate_model(data_list, model_type="GCN", epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    labels = [int(data.y.item()) for data in data_list]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    acc_scores, f1_scores = [], []
    print(f" [{model_type}] 모델 5-Fold 교차 검증 시작 ")

    for fold, (train_idx, test_idx) in enumerate(skf.split(data_list, labels)):
        train_data = [data_list[i] for i in train_idx]
        test_data = [data_list[i] for i in test_idx]
        
        train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
        
        input_dim = data_list[0].num_features 
        
        if model_type == "GCN":
            model = BrainGCN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
        elif model_type == "GIN":
            model = BrainGIN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
            
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = torch.nn.CrossEntropyLoss()
        
        # 학습
        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()
                
        # 평가
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                preds.extend(out.argmax(dim=1).cpu().numpy())
                trues.extend(batch.y.cpu().numpy())
                
        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds, average='macro')
        acc_scores.append(acc)
        f1_scores.append(f1)
        print(f"Fold {fold+1} | Accuracy: {acc:.4f} | F1 Score: {f1:.4f}")
        
    print(f" [{model_type}] 최종 평균 -> Accuracy: {np.mean(acc_scores):.4f} | F1: {np.mean(f1_scores):.4f}\n")
    return np.mean(acc_scores), np.mean(f1_scores)

if __name__ == "__main__":
    V8_DATA_DIR = "data_Schaefer_Adult_Top20"

    all_data = load_v8_data(V8_DATA_DIR)

    gcn_acc, gcn_f1 = evaluate_model(all_data, model_type="GCN", epochs=50)
    
    gin_acc, gin_f1 = evaluate_model(all_data, model_type="GIN", epochs=50)
    
    print("====================================================")
    print(f"- GCN | Accuracy: {gcn_acc:.4f} | F1: {gcn_f1:.4f}")
    print(f"- GIN | Accuracy: {gin_acc:.4f} | F1: {gin_f1:.4f}")
    print("====================================================")

 [data_Schaefer_Adult_Top20] 경로에서 V8 전체 데이터 로드 중...
✅ 총 258명 전체 데이터 로드 완료!

 [GCN] 모델 5-Fold 교차 검증 시작 
Fold 1 | Accuracy: 0.5385 | F1 Score: 0.3500
Fold 2 | Accuracy: 0.5769 | F1 Score: 0.4359
Fold 3 | Accuracy: 0.5577 | F1 Score: 0.3580
Fold 4 | Accuracy: 0.5490 | F1 Score: 0.3544
Fold 5 | Accuracy: 0.5490 | F1 Score: 0.3544
 [GCN] 최종 평균 -> Accuracy: 0.5542 | F1: 0.3706

 [GIN] 모델 5-Fold 교차 검증 시작 
Fold 1 | Accuracy: 0.4808 | F1 Score: 0.4462
Fold 2 | Accuracy: 0.5577 | F1 Score: 0.3944
Fold 3 | Accuracy: 0.5385 | F1 Score: 0.3500
Fold 4 | Accuracy: 0.5686 | F1 Score: 0.4006
Fold 5 | Accuracy: 0.5490 | F1 Score: 0.4848
 [GIN] 최종 평균 -> Accuracy: 0.5389 | F1: 0.4152

- GCN | Accuracy: 0.5542 | F1: 0.3706
- GIN | Accuracy: 0.5389 | F1: 0.4152
